In [ ]:
import pandas as pd
import requests
import time

print("🏈 Iniciando extracción masiva de estadísticas NFL 2026...")
temporada = 2026
game_ids = []
lista_estadisticas = []

# ==========================================
# FASE 1: Obtener todos los IDs de los partidos
# ==========================================
for semana in range(1, 19):
    print(f"📅 Recolectando calendario Semana {semana}...")
    url_calendario = f"https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/scoreboard?dates={temporada}&seasontype=2&week={semana}"
    
    try:
        res = requests.get(url_calendario, timeout=10)
        if res.status_code == 200:
            datos = res.json()
            for evento in datos.get('events', []):
                game_ids.append({
                    'ID_Partido': evento.get('id'),
                    'Semana': semana
                })
    except Exception as e:
        print(f"Error en semana {semana}: {e}")
        
    time.sleep(0.5)

print(f"✅ Se encontraron {len(game_ids)} partidos. Iniciando extracción de Boxscores (esto tomará varios minutos)...")

# ==========================================
# FASE 2: Extraer estadísticas jugada por jugada
# ==========================================
contador = 1
total_partidos = len(game_ids)

for partido in game_ids:
    id_partido = partido['ID_Partido']
    semana = partido['Semana']
    url_summary = f"https://site.web.api.espn.com/apis/site/v2/sports/football/nfl/summary?event={id_partido}"
    
    print(f"📊 Procesando partido {contador}/{total_partidos} (ID: {id_partido})...")
    
    try:
        res = requests.get(url_summary, timeout=10)
        if res.status_code == 200:
            datos = res.json()
            boxscore = datos.get('boxscore', {}).get('players', [])
            
            # Recorrer a los dos equipos que jugaron
            for equipo in boxscore:
                id_equipo = equipo.get('team', {}).get('id', 'N/A')
                
                # Recorrer las categorías (passing, rushing, receiving, defensive, etc.)
                for categoria in equipo.get('statistics', []):
                    nombre_categoria = categoria.get('name', 'N/A')
                    columnas_stat = categoria.get('keys', []) # Ej: ['YDS', 'TD', 'INT']
                    
                    # Recorrer a los jugadores de esa categoría
                    for atleta in categoria.get('athletes', []):
                        stats_valores = atleta.get('stats', [])
                        
                        fila_stat = {
                            'ID_Partido': id_partido,
                            'Semana': semana,
                            'ID_Equipo': id_equipo,
                            'ID_Jugador': atleta.get('athlete', {}).get('id', 'N/A'),
                            'Nombre_Jugador': atleta.get('athlete', {}).get('displayName', 'Desconocido'),
                            'Categoria_Juego': nombre_categoria.capitalize()
                        }
                        
                        # Unimos el nombre de la estadística con el valor del jugador
                        for col, val in zip(columnas_stat, stats_valores):
                            # Creamos columnas dinámicas como 'Passing_YDS' o 'Defensive_SACKS'
                            fila_stat[f"{nombre_categoria}_{col}"] = val
                            
                        lista_estadisticas.append(fila_stat)
    except Exception as e:
        print(f"⚠️ Error al procesar el partido {id_partido}: {e}")
        
    contador += 1
    time.sleep(1) # Pausa estratégica vital para no ser bloqueados

# ==========================================
# FASE 3: Guardar y Exportar a CSV
# ==========================================
print("💾 Convirtiendo datos a tabla...")
df_estadisticas = pd.DataFrame(lista_estadisticas)

# Guardamos el CSV
nombre_archivo = 'nfl_boxscores_2026.csv'
df_estadisticas.to_csv(nombre_archivo, index=False, encoding='utf-8')

print(f"🚀 ¡Proceso finalizado exitosamente! Se guardaron {len(df_estadisticas)} filas de estadísticas en '{nombre_archivo}'.")
print(f"El archivo pesa aproximadamente {round(os.path.getsize(nombre_archivo) / (1024 * 1024), 2)} MB.")

🏈 Iniciando extracción masiva de estadísticas NFL 2026...
📅 Recolectando calendario Semana 1...
📅 Recolectando calendario Semana 2...
📅 Recolectando calendario Semana 3...
📅 Recolectando calendario Semana 4...
📅 Recolectando calendario Semana 5...
📅 Recolectando calendario Semana 6...
📅 Recolectando calendario Semana 7...
📅 Recolectando calendario Semana 8...
📅 Recolectando calendario Semana 9...
📅 Recolectando calendario Semana 10...
📅 Recolectando calendario Semana 11...
